# Cleaning Data

The explainer introduced five common data quality problems (missing values, duplicates, inconsistent formatting, impossible values, and outliers) and the judgement calls involved in deciding what to do about each one. We also saw that the reason data is missing often matters more than the fact that it is missing, and that not every problem is ours to fix.

Now we will apply those ideas. The combined shipments dataset from Notebook 1 is functional, but the warehouse inventory is a mess: inconsistent product names, missing values, and duplicate rows. This is typical of data that has been entered by multiple people over time without strict validation rules. Before any analysis can happen, the data needs cleaning.

By the end of this notebook we will be able to:

- Detect and quantify missing values with `.isnull()` and `.isnull().sum()`
- Choose between `dropna` and `fillna` based on the situation
- Find and remove duplicate rows with `.duplicated()` and `.drop_duplicates()`
- Normalise inconsistent strings with `.str.strip()`, `.str.lower()`, and `.str.replace()`
- Parse and convert dates with `pd.to_datetime`
- Fix data types with `.astype()`

In [ ]:
%pip install -q pandas

In [ ]:
import pandas as pd

---

## 1. Loading the warehouse inventory

The warehouse inventory CSV has several known problems: missing product names, missing quantities, duplicate rows, and inconsistent naming (e.g. "Widget A", "widget-a", "WIDGET A", "Widget  A" all refer to the same product).

We might recognise these from the explainer's categories. The inconsistent naming is a formatting problem that will break any grouping or filtering operation. The missing values need different handling depending on which column they affect. The duplicates inflate counts. All of these need to be resolved before the data can support reliable analysis.

In [ ]:
inventory = pd.read_csv("../data/warehouse_inventory.csv")
print(f"Shape: {inventory.shape}")
inventory.head(10)

In [ ]:
print(inventory.dtypes)
print()
print(inventory.info())

Notice that `quantity` is `object` (string) instead of integer. That is because some rows have empty strings in that column, which prevents pandas from parsing it as numeric. This is a common gotcha: a single non-numeric value in a column forces the entire column to be stored as text. We will fix this after handling the missing values.

---

## 2. Detecting missing values

Before we can decide how to handle missing data, we need to know what is missing and how much. The `.isnull()` method returns a DataFrame of booleans: `True` where a value is missing, `False` where it is present. Chaining `.sum()` counts the `True` values per column.

In [ ]:
# pandas only sees NaN as null. Empty strings "" are not detected by .isnull().
# We need to replace empty strings with NaN first.
inventory = inventory.replace("", pd.NA)

print(inventory.isnull().sum())
print(f"\nTotal rows: {len(inventory)}")
print(f"Percentage missing product_name: {inventory['product_name'].isnull().mean():.1%}")
print(f"Percentage missing quantity: {inventory['quantity'].isnull().mean():.1%}")

### Choosing a strategy: `dropna` vs `fillna`

The explainer presented four strategies for handling missing data: delete the rows, fill in a substitute, flag and include, or go back to the source. In pandas, the first two map directly to `dropna` and `fillna`.

| Method | Use when | Example |
|--------|----------|---------|
| `dropna()` | The row is useless without the missing value, or we have enough data to lose some rows | Drop rows where `product_name` is missing (we cannot analyse an unnamed product) |
| `fillna(value)` | We can supply a reasonable default, or the missing value can be inferred | Fill missing `quantity` with `0` or the column median |

There is no universal rule. The right choice depends on what the data will be used for and how much we can afford to lose. The key is to make a deliberate decision for each column rather than applying the same fix everywhere.

In [ ]:
# Strategy:
# - Drop rows where product_name is missing (we cannot identify the product)
# - Fill missing quantity with 0 (treat as "no stock recorded")

before = len(inventory)
inventory = inventory.dropna(subset=["product_name"])
print(f"Dropped {before - len(inventory)} rows with missing product_name")
print(f"Rows remaining: {len(inventory)}")

# Convert quantity to numeric first, then fill nulls
inventory["quantity"] = pd.to_numeric(inventory["quantity"], errors="coerce")
inventory["quantity"] = inventory["quantity"].fillna(0).astype(int)

print(f"\nNull check after cleaning:")
print(inventory.isnull().sum())

---

## 3. Detecting and removing duplicates

The explainer described how duplicates arise: the same entity entered twice under slightly different details, or a system recording both an initial submission and a retry. In our inventory, some rows are exact copies, the simplest kind of duplicate. The `.duplicated()` method flags rows that are exact copies of earlier rows. By default it marks the second (and subsequent) occurrence as `True`.

In [ ]:
duplicates = inventory.duplicated()
print(f"Duplicate rows found: {duplicates.sum()}")

# Look at a few duplicates
inventory[duplicates].head()

In [ ]:
before = len(inventory)
inventory = inventory.drop_duplicates()
print(f"Removed {before - len(inventory)} duplicate rows")
print(f"Rows remaining: {len(inventory)}")

We can also check for duplicates on a subset of columns. For example, if each warehouse should only have one row per product_id:

```python
inventory.duplicated(subset=["warehouse_id", "product_id"])
```

---

## 4. String normalisation

This is the inconsistent formatting problem from the explainer in action. The product names in the inventory are recorded in different ways: "Widget A", "widget-a", "WIDGET A", and "Widget  A" (with a double space). They all refer to the same product, but pandas treats each spelling as a different value. Any grouping, filtering, or counting operation will give wrong results until we normalise these to a single canonical form.

The key string methods are:

| Method | What it does |
|--------|-------------|
| `.str.strip()` | Removes leading and trailing whitespace |
| `.str.lower()` | Converts to lowercase |
| `.str.replace(old, new)` | Replaces substrings |

These methods are accessed through the `.str` accessor on a pandas Series.

In [ ]:
# Before normalisation: how many distinct product names?
print(f"Unique product names before: {inventory['product_name'].nunique()}")
print()
print(inventory["product_name"].value_counts().head(20))

In [ ]:
# Normalise: lowercase, replace hyphens with spaces, collapse multiple spaces, strip
inventory["product_name"] = (
    inventory["product_name"]
    .str.lower()
    .str.replace("-", " ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

print(f"Unique product names after: {inventory['product_name'].nunique()}")
print()
print(inventory["product_name"].value_counts())

The chain of operations works left to right:
1. `.str.lower()` converts everything to lowercase ("WIDGET A" becomes "widget a")
2. `.str.replace("-", " ")` turns hyphens into spaces ("widget-a" becomes "widget a")
3. `.str.replace(r"\s+", " ", regex=True)` collapses multiple spaces into one ("widget  a" becomes "widget a")
4. `.str.strip()` removes any leading or trailing whitespace

After normalisation, the number of unique product names drops to match the actual number of distinct products. This is a quick fix with a clear rationale — exactly the kind of problem the explainer said an analyst should handle during cleaning and document.

---

## 5. Date parsing and type conversion

The `last_restock_date` column is stored as a string. We convert it to a proper datetime type using `pd.to_datetime`. This allows date-based filtering, sorting, and extraction (e.g. pulling out the month or calculating the number of days since the last restock).

In [ ]:
print(f"Before: {inventory['last_restock_date'].dtype}")

inventory["last_restock_date"] = pd.to_datetime(inventory["last_restock_date"], format="%Y-%m-%d")

print(f"After:  {inventory['last_restock_date'].dtype}")
inventory["last_restock_date"].head()

### Fixing data types with `.astype()`

The `.astype()` method converts a column to a specified type. Common conversions:

| From | To | Method |
|------|----|--------|
| `object` (string) | `int` | `.astype(int)` |
| `object` (string) | `float` | `.astype(float)` |
| `float` | `int` | `.astype(int)` (only if no NaN) |
| `int` | `str` | `.astype(str)` |

We already converted `quantity` to int when we filled nulls. Let us verify the final types to make sure everything is in order.

In [ ]:
print(inventory.dtypes)
print()
inventory.head()

---

## 6. Putting it all together: before and after

Each cleaning step above addressed one problem in isolation. In practice, we run them as a pipeline — a sequence of transformations applied to the raw data in one pass. This makes the cleaning reproducible: anyone can re-run the cell and get the same result from the same input.

Let us reload the raw data and run the full cleaning pipeline in one cell to see the complete transformation.

In [ ]:
# Full cleaning pipeline
raw = pd.read_csv("../data/warehouse_inventory.csv")
print(f"Raw: {raw.shape[0]} rows, {raw['product_name'].nunique()} unique product names")

clean = (
    raw
    .replace("", pd.NA)                         # mark empties as null
    .dropna(subset=["product_name"])             # drop rows with no product name
    .assign(
        quantity=lambda df: pd.to_numeric(df["quantity"], errors="coerce").fillna(0).astype(int),
        product_name=lambda df: (
            df["product_name"]
            .str.lower()
            .str.replace("-", " ", regex=False)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        ),
        last_restock_date=lambda df: pd.to_datetime(df["last_restock_date"], format="%Y-%m-%d"),
    )
    .drop_duplicates()
    .reset_index(drop=True)
)

print(f"Clean: {clean.shape[0]} rows, {clean['product_name'].nunique()} unique product names")
print(f"Nulls remaining: {clean.isnull().sum().sum()}")
clean.head()

---

## Exercises

Now it is your turn. These exercises ask you to apply cleaning techniques and, just as importantly, think about *why* you are choosing one approach over another.

Complete each exercise in the code cell provided.

### Exercise 1: Analyse missing data patterns

Reload `warehouse_inventory.csv` (with empty strings replaced by `pd.NA`). For each column, calculate and print:
- The count of missing values
- The percentage of missing values (to 1 decimal place)
- Whether the missing values appear to be random or systematic (add a comment with your reasoning)

Then use `dropna(how="any")` and `dropna(how="all")` separately and compare the results. Which approach loses more data? Why?

In [ ]:
# Your code here


### Exercise 2: Deduplicate with a twist

Reload the raw inventory data and replace empty strings with `pd.NA`. Before removing duplicates, normalise the `product_name` column (lowercase, hyphens to spaces, collapse whitespace, strip). Then check for duplicates again.

How many duplicates are there before normalisation? How many after? Why does normalisation affect the duplicate count?

In [ ]:
# Your code here


### Exercise 3: Clean the combined shipments data

Load the three shipment files (Asia CSV, Europe CSV, Americas JSON), normalise them as in Notebook 1, and concatenate them. Then:

1. Convert `ship_date` to a proper datetime column using `pd.to_datetime`
2. Convert `status` to lowercase and strip whitespace
3. Check for and report any duplicate `shipment_id` values
4. Print the final `.dtypes` and `.isnull().sum()`

In [ ]:
# Your code here


### Exercise 4: `fillna` strategies compared

Reload the raw inventory and replace empty strings with `pd.NA`. Convert the `quantity` column to numeric (using `pd.to_numeric` with `errors="coerce"`). Then create three versions of the DataFrame:

1. `v1`: fill missing quantities with `0`
2. `v2`: fill missing quantities with the column **median**
3. `v3`: fill missing quantities with the **mean quantity for that warehouse** (use `.groupby("warehouse_id")["quantity"].transform("mean")`)

For each version, print the mean and standard deviation of the `quantity` column. Which approach changes the distribution the least?

In [ ]:
# Your code here


---

## Summary

We started with a messy inventory dataset and worked through the main categories of data quality problems: missing values, duplicates, inconsistent formatting, and wrong data types. For each one, we made a deliberate choice about how to handle it — drop or fill, normalise or leave, convert or keep as-is.

In this notebook we:

- Detected missing values with `.isnull().sum()` and replaced empty strings with `pd.NA`
- Used **`dropna`** to remove rows with critical missing data and **`fillna`** to supply defaults where appropriate
- Found and removed **duplicate rows** with `.duplicated()` and `.drop_duplicates()`
- Normalised inconsistent strings using `.str.lower()`, `.str.replace()`, and `.str.strip()`
- Converted date strings to proper datetime objects with **`pd.to_datetime`**
- Fixed column types with `.astype()` and `pd.to_numeric`

The data is now clean, but it is still in its raw shape. In the next notebook we will transform it: creating derived columns, binning values, aggregating, and reshaping for analysis and presentation.